# Database browser (SQLite)

**What it does**
Opens the app's own SQLite database (`storage/database.sqlite`, or `KCT_DB_PATH`) read-only and shows its contents as tables in the notebook: all tables with their row counts, the columns of a table, every case interpretation (one interpretation run), the test steps of one run, and the result of any SQL you type.

**Why**
The ING machine has no web page or desktop app for looking into a database, so this notebook is the database GUI. Use it to check what the interpretation stored, compare runs, or find an ID. It opens the file with `mode=ro`, so nothing can be changed by accident.

**How to use**
1. The app must be using SQLite: `KCT_DB_DRIVER=sqlite`, and at least one interpretation run done with that setting (`POST /api/v1/test-steps`, or `boot().interpretations.create_and_generate(...)`). With the default `memory` driver there is no file to look at.
2. Open this notebook from the project's `notebooks/` folder and run all cells.
3. In "Test steps of one case interpretation", paste an `id` from the table above it and run that cell again. Left empty, it shows the newest run.
4. In "Any query", write your own `SELECT` and run the cell. `q("... WHERE x = ?", value)` passes values safely.

**Needs**: pandas (usually already on the workbench). To only check that SQLite works, use `03_sqlite_test.ipynb` instead.

## Setup

Finds the project folder, reads which database file the app uses, and defines `q(sql, *params)`: runs a read-only query and returns the result as a table. If this cell says MISSING, see step 1 above.

In [ ]:
import os, sys, sqlite3
from pathlib import Path

import pandas as pd

# The project folder: the parent of notebooks/. Change it if you moved the notebook.
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from services.config import settings

DB = settings().db_path  # KCT_DB_PATH, default storage/database.sqlite
print(DB, "exists" if DB.is_file() else "MISSING: set KCT_DB_DRIVER=sqlite and run an interpretation first")

pd.set_option("display.max_colwidth", 200)


def q(sql, *params):
    """Run a read-only query and show the result as a table."""
    with sqlite3.connect(f"file:{DB}?mode=ro", uri=True) as con:
        return pd.read_sql_query(sql, con, params=params)

## Tables and row counts

In [ ]:
tables = q("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")
pd.DataFrame(
    {"table": t, "rows": q(f'SELECT COUNT(*) AS n FROM "{t}"').n[0]} for t in tables.name
)

## Columns of one table

In [ ]:
q("SELECT name, type, \"notnull\", pk FROM pragma_table_info(?)", "test_steps")

## Case interpretations, newest first

In [ ]:
q("""
SELECT ci.id, c.reference AS case_reference, ci.provider, ci.model, ci.created_at,
       (SELECT COUNT(*) FROM test_steps ts WHERE ts.case_interpretation_id = ci.id) AS steps
FROM case_interpretations ci JOIN cases c ON c.id = ci.case_id
ORDER BY ci.created_at DESC
""")

## Test steps of one case interpretation

Paste an `id` from the table above.

In [ ]:
CASE_INTERPRETATION_ID = ""   # paste an id from the table above; empty: the newest run

if not CASE_INTERPRETATION_ID:
    newest = q("SELECT id FROM case_interpretations ORDER BY created_at DESC LIMIT 1")
    CASE_INTERPRETATION_ID = newest.id[0] if len(newest) else ""

q("""
SELECT position, reference, title, status, description, acceptance_criteria, refusal_criteria
FROM test_steps WHERE case_interpretation_id = ? ORDER BY position
""", CASE_INTERPRETATION_ID)

## Any query

In [ ]:
q("SELECT * FROM cases ORDER BY created_at DESC LIMIT 20")